In [1]:
import numpy as np 
import pandas as pd
import random

In [ ]:
def generate_data():

    INTENTS = ["attack", "retreat", "reposition", "idle"]
    ACTIONS = ["center_flank", "from_left_flank", "from_right_flank", "moving_back"]

    TYPE_WEIGHTS = {"TANK": 1.0, "IFV": 0.8, "APC": 0.7}
    ACTION_WEIGHTS = {
        "center_flank": 1.0,
        "from_left_flank": 0.9,
        "from_right_flank": 0.9,
        "moving_back": 0.3
    }
    INTENT_WEIGHTS = {
        "attack": 1.0,
        "reposition": 0.7,
        "retreat": 0.3,
        "idle": 0.1
    }

    MAX_DIST = 1000

    CLASSES = ["TANK", "IFV", "APC"]


    # ---------------------------------------------
    cls = random.choice(CLASSES)
    type_score = TYPE_WEIGHTS[cls]
    # ---------------------------------------------

    X = random.uniform(-200, 200)
    Y = random.uniform(5, 500)

    vx = random.uniform(-5, 5)
    vy = random.uniform(-5, 5)

    X += np.random.normal(0, 5)
    Y += np.random.normal(0, 5)

    distance = np.sqrt(X**2 + Y**2)
    distance_score = max(0, 1 - distance / MAX_DIST)
    distance_score += np.random.normal(0, 0.02)
    distance_score = np.clip(distance_score, 0, 1)    

    # ---------------------------------------------

    future_Y = Y + vy * 2
    future_dist_score = max(0, 1 - future_Y / MAX_DIST)
    
    future_dist_score += np.random.normal(0, 0.02)
    future_dist_score = np.clip(future_dist_score, 0, 1)

    # ---------------------------------------------

    action = random.choice(ACTIONS)
    action_score = ACTION_WEIGHTS[action]

    action_proba = random.uniform(0.3, 1.0)
    action_proba += np.random.normal(0, 0.05)
    action_proba = np.clip(action_proba, 0, 1)

    action_score *= action_proba

    # ---------------------------------------------
   

    if abs(vx) + abs(vy) < 0.2:
        direction_score = 0.5
    elif vy < 0:
        direction_score = 1.0
    elif abs(vx) > abs(vy):
        direction_score = 0.7
    else:
        direction_score = 0.3

    # ---------------------------------------------

    conf = random.uniform(0.5, 1.0)
    conf += np.random.normal(0, 0.05)
    conf = np.clip(conf, 0, 1)

    # ---------------------------------------------

    intent = random.choice(INTENTS)
    intent_score = INTENT_WEIGHTS[intent]

    threat = (
        0.2 * type_score +
        0.15 * distance_score +
        0.15 * future_dist_score +
        0.15 * action_score +
        0.1 * direction_score +
        0.1 * conf +
        0.2 * intent_score
    )

    threat += np.random.normal(0, 0.03)
    threat = np.clip(threat, 0, 1)

    return [type_score, distance_score, future_dist_score, action_score, direction_score, conf, intent_score, threat]


In [3]:
data = [generate_data() for _ in range(100_000)]

In [4]:
columns = [
    "class_score",              
    "distance_score",
    "future_dist_score",
    "action",
    "direction_score",
    "confidence",
    "intent",
    "threat"
]

data = pd.DataFrame(data, columns=columns)

In [5]:
data

,class_score,distance_score,future_dist_score,action,direction_score,confidence,intent,threat
0,1.0,0.613639,0.632597,0.228065,1.0,0.572076,0.1,0.597933
1,0.8,0.784872,0.875178,0.839524,1.0,0.619044,0.3,0.781948
2,0.7,0.698952,0.743294,0.798072,1.0,0.625434,1.0,0.893746
3,1.0,0.640992,0.642332,0.132402,0.3,0.870379,0.1,0.557315
4,0.8,0.807733,0.777699,0.661243,0.7,0.772051,0.3,0.665039
...,...,...,...,...,...,...,...,...
99995,0.8,0.756411,0.756640,0.485782,1.0,0.639526,0.3,0.750454
99996,0.8,0.765817,0.797266,0.402158,0.7,0.731030,0.3,0.625890
99997,0.7,0.723765,0.743735,0.220021,0.3,0.461820,0.3,0.583581
99998,1.0,0.816114,0.918378,0.105402,1.0,0.763687,0.3,0.726183


In [6]:
min(data['distance_score'])

0.4078770766268297

In [7]:
data.to_csv('threat_data.csv')